## Codigo para obtener los puntajes

In [ ]:
import pandas as pd
import numpy as np


file_path = 'Formulario Importante.csv'

# Nombres de las secciones 
new_section_names = [
    "Integridad", "Consistencia", "Precisión", "Tolerancia a Errores",
    "Simplicidad", "Modularidad", "Expansibilidad", "Instrumentación",
    "Auto-descriptividad", "Eficiencia de ejecución", "Concisión",
    "Operabilidad", "Facilidad de Aprendizaje", "Comunicatividad",
    "Estandarización de datos y estructuras"
]

# Definición de los "temas" (sub-secciones)
subsection_map = {
    "Tolerancia a Errores": {
        "Entrada señas -> texto": (0, 5),  # Preguntas 1-5
        "Entrada texto -> señas": (5, 10) # Preguntas 6-10
    },
    "Expansibilidad": {
        "Expansión del almacenamiento": (0, 2), # Preguntas 1-2
        "Extensibilidad": (2, 5)              # Preguntas 3-5
    },
    "Instrumentación": {
        "Pruebas de módulos": (0, 2),        # Preguntas 1-2
        "Pruebas de integración": (2, 4),      # Preguntas 3-4
        "Pruebas del sistema completo": (4, 6) # Preguntas 5-6
    },
    "Auto-descriptividad": {
        "Cantidad de comentarios": (0, 1),           # Pregunta 1
        "Calidad y utilidad de los comentarios": (1, 4), # Preguntas 2-4
        "Claridad del código fuente": (4, 6)         # Preguntas 5-6
    },
    "Eficiencia de ejecución": {
        "Asignación de rendimiento en el diseño": (0, 1), # Pregunta 1
        "Eficiencia en bucles y expresiones": (1, 4),  # Preguntas 2-4
        "Eficiencia de uso de datos": (4, 6)         # Preguntas 5-6
    },
    "Comunicatividad": {
        "Entrada del usuario (Input Interface)": (0, 4), # Preguntas 1-4
        "Salida del usuario (Output Interface)": (4, 7)  # Preguntas 5-7
    }
}


# --- Carga de Datos ---
print(f"Cargando archivo: {file_path}...")
try:
    df = pd.read_csv(file_path)
except UnicodeDecodeError:
    print("UTF-8 falló. Intentando con codificación 'latin1'.")
    df = pd.read_csv(file_path, encoding='latin1')

# --- Procesamiento y Cálculo ---


df_questions = df.drop(columns=['Marca temporal'], errors='ignore')
df_numeric = df_questions.apply(pd.to_numeric, errors='coerce')

# Identificar secciones
question_cols = df_numeric.columns.tolist()
section_start_indices = [i for i, col in enumerate(question_cols) if col.strip().startswith('1.-')]

print("\n--- Cálculo de Promedios por Sección (Promedio de Promedios de Temas) ---")

all_section_averages_list = []

if not section_start_indices:
    print("No se detectaron secciones que comiencen con '1.-'.")
else:
    section_start_indices.append(len(question_cols))
    num_detected_sections = len(section_start_indices) - 1

    if num_detected_sections != len(new_section_names):
        print(f"Advertencia: Se detectaron {num_detected_sections} secciones,")
        print(f"pero se proporcionaron {len(new_section_names)} nombres.")

    # Loop principal para iterar sobre cada sección
    for i in range(num_detected_sections):
        
        section_name = new_section_names[i] if i < len(new_section_names) else f"Sección {i+1}"
        
        start_index = section_start_indices[i]
        end_index = section_start_indices[i+1]
        section_cols = question_cols[start_index:end_index]
        
        print(f"\n--- {section_name} ---")

        # Verificar si esta sección tiene "temas" (sub-secciones) definidos
        if section_name in subsection_map:
            sub_map = subsection_map[section_name]
            
            # Lista para guardar los promedios de los temas de ESTA sección
            theme_averages_list = []
            
            # Calcular el promedio de cada tema
            for sub_name, (sub_start_idx, sub_end_idx) in sub_map.items():
                try:
                    sub_section_cols = section_cols[sub_start_idx:sub_end_idx]
                    
                    if not sub_section_cols:
                         print(f"  > Tema: {sub_name}: (No hay columnas en el rango {sub_start_idx}:{sub_end_idx})")
                         continue

                    # Promedio del tema (ponderado por respuestas)
                    sub_mean = df_numeric[sub_section_cols].values.mean()
                    print(f"  > Tema: {sub_name}: {sub_mean:.2f}")
                    # Guardar el promedio del tema
                    theme_averages_list.append(sub_mean)
                
                except Exception as e:
                    print(f"  > Error al calcular el tema '{sub_name}': {e}")

            # Calcular el promedio DE LOS PROMEDIOS de los temas
            if theme_averages_list:
                section_mean_of_themes = np.mean(theme_averages_list)
                print(f"Promedio (de los promedios de temas) {section_name}: {section_mean_of_themes:.2f}")
                # Guardar este promedio para el cálculo general
                all_section_averages_list.append(section_mean_of_themes)
            else:
                print(f"No se pudieron calcular temas para {section_name}.")
                all_section_averages_list.append(np.nan) # Añadir NaN si falló

        else:
            # Si no hay temas, solo calcular el promedio normal de la sección
            section_mean = df_numeric[section_cols].values.mean()
            print(f"Promedio de la sección: {section_mean:.2f}")
            # Guardar este promedio para el cálculo general
            all_section_averages_list.append(section_mean)


    overall_average_of_sections = np.nanmean(all_section_averages_list)
    print(f"\n=================================================")
    print(f"Promedio general (de los 15 promedios de sección): {overall_average_of_sections:.2f}")



print("--- Proceso completado ---")

Cargando archivo: Formulario Importante.csv...
Archivo cargado exitosamente.

--- Cálculo de Promedios por Sección (Promedio de Promedios de Temas) ---

--- Integridad ---
Promedio de la sección: 8.67

--- Consistencia ---
Promedio de la sección: 8.58

--- Precisión ---
Promedio de la sección: 9.33

--- Tolerancia a Errores ---
  > Tema: Entrada señas -> texto: 6.63
  > Tema: Entrada texto -> señas: 7.90
Promedio (de los promedios de temas) Tolerancia a Errores: 7.27

--- Simplicidad ---
Promedio de la sección: 9.07

--- Modularidad ---
Promedio de la sección: 9.28

--- Expansibilidad ---
  > Tema: Expansión del almacenamiento: 8.83
  > Tema: Extensibilidad: 8.83
Promedio (de los promedios de temas) Expansibilidad: 8.83

--- Instrumentación ---
  > Tema: Pruebas de módulos: 8.17
  > Tema: Pruebas de integración: 9.25
  > Tema: Pruebas del sistema completo: 9.00
Promedio (de los promedios de temas) Instrumentación: 8.81

--- Auto-descriptividad ---
  > Tema: Cantidad de comentarios: 8.8